In [1]:
import pandas as pd

# Load the datasets
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

# Display basic information and shape
print(f"Train set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

print("\nTrain Columns:", train_df.columns.tolist())
print("Test Columns:", test_df.columns.tolist())

# Preview the first few rows of the training data
display(train_df.head())

Train set shape: (1039, 7)
Test set shape: (260, 2)

Train Columns: ['id', 'text', 'label', 'word_count', 'char_count', 'avg_word_length', 'punctuation_density']
Test Columns: ['id', 'text']


,id,text,label,word_count,char_count,avg_word_length,punctuation_density
0,1,Artificial intelligence is rapidly transformin...,ai,9,76,7.555556,0.111111
1,2,The delivery drone dropped a package on the do...,ai,9,53,5.000000,0.111111
2,3,No man has a good enough memory to be a succes...,human,12,56,3.750000,0.083333
3,4,The autonomous car adjusted its speed as traff...,ai,9,56,5.333333,0.111111
4,5,"Power tends to corrupt, and absolute power cor...",human,9,63,6.111111,0.222222


In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# Map labels to numeric values: 'human' -> 0, 'ai' -> 1
label_map = {'human': 0, 'ai': 1}
train_df['target'] = train_df['label'].map(label_map)

X = train_df['text'].values
y = train_df['target'].values
X_test = test_df['text'].values

# Set up Stratified K-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

# Vectorizer configuration
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)

# Stratified CV loop
f1_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    # Transform features
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)
    X_test_vec = vectorizer.transform(X_test)

    # Model initialization and training
    model = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
    model.fit(X_train_vec, y_train)

    # Validation predictions
    val_pred = model.predict(X_val_vec)
    oof_preds[val_idx] = val_pred

    # Accumulate test predictions (soft probabilities)
    test_preds += model.predict_proba(X_test_vec)[:, 1] / skf.n_splits

    fold_f1 = f1_score(y_val, val_pred)
    f1_scores.append(fold_f1)
    print(f"Fold {fold + 1} F1 Score: {fold_f1:.4f}")

print(f"\nMean OOF F1 Score: {np.mean(f1_scores):.4f}")

Fold 1 F1 Score: 0.9488
Fold 2 F1 Score: 0.9615
Fold 3 F1 Score: 0.9434
Fold 4 F1 Score: 0.9395
Fold 5 F1 Score: 0.9320

Mean OOF F1 Score: 0.9451


In [3]:
# Convert probability predictions to binary labels
final_test_labels = (test_preds >= 0.5).astype(int)

# Generate submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'label': final_test_labels
})

# Save submission file
submission_file_path = 'submission.csv'
submission.to_csv(submission_file_path, index=False)
print(f"Submission file successfully saved to {submission_file_path}")

# Display preview of submission and validation details
display(submission.head())
print(f"Submission shape: {submission.shape}")
print(f"Label distributions:\n{submission['label'].value_counts()}")

Submission file successfully saved to submission.csv


,id,label
0,1,0
1,2,1
2,3,0
3,4,1
4,5,0


Submission shape: (260, 2)
Label distributions:
label
0    130
1    130
Name: count, dtype: int64
